In [16]:
# ==============================================================================
# KHỞI TẠO MÔI TRƯỜNG & ĐỌC DỮ LIỆU TỪ KAGGLE (ĐÃ QUA LÀM SẠCH VÀ SPLIT)
# ==============================================================================
import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

print("--- ĐANG TẢI DỮ LIỆU ---")
# Bạn đổi lại đường dẫn trỏ đúng vào thư mục dataset trên Kaggle nhé
DATA_DIR = '/kaggle/input/datasets/phn217/temp-ds/'

df_train = pd.read_csv(f"{DATA_DIR}train_cleaned_ds108.csv")
df_test = pd.read_csv(f"{DATA_DIR}test_cleaned_ds108.csv")

print(f"-> Kích thước tập Train: {df_train.shape}")
print(f"-> Kích thước tập Test: {df_test.shape}")

--- ĐANG TẢI DỮ LIỆU ---
-> Kích thước tập Train: (4880, 45)
-> Kích thước tập Test: (1222, 45)


# 1. Kỹ thuật Đặc trưng (Feature Engineering)

<div style="font-size: 17px;">

Áp dụng Target Encoding chuyển đổi các cột dữ liệu dạng chữ sang định dạng số học để mô hình có thể tính toán. Phương pháp này mã hóa chữ thành số dựa trên sự tương quan với biến mục tiêu, cực kỳ tối ưu cho các thuật toán dạng Cây (Tree-based).

</div>

In [17]:
print("--- TARGET ENCODING ---")

# 1. Mã hóa Biến Mục Tiêu (Target) sang số (0, 1, 2...)
print("1. Đang mã hóa biến mục tiêu (Label Encoding)...")
label_encoder = LabelEncoder()
df_train['target_category'] = label_encoder.fit_transform(df_train['category_name'])
df_test['target_category'] = label_encoder.transform(df_test['category_name'])

# 2. Lựa chọn các biến Features bằng Target Encoding 
cat_features = [
    'region', 
    'publish_day_of_week',
    'definition', 
    'license', 
    'default_language'
]

# Sử dụng tham số smoothing để làm mượt, tránh overfitting cho các danh mục hiếm (như channel ít video)
target_encoder = ce.TargetEncoder(cols=cat_features, smoothing=10)

print(f"2. Đang học quy luật (Fit) và mã hóa tập Train trên {len(cat_features)} cột...")
df_train_encoded = target_encoder.fit_transform(df_train[cat_features], df_train['target_category'])

print("3. Đang áp dụng quy luật (Transform) lên tập Test...")
df_test_encoded = target_encoder.transform(df_test[cat_features])

# 3. Gắn lại vào DataFrame gốc và dọn dẹp các cột chữ cũ
df_train = df_train.drop(columns=cat_features).join(df_train_encoded)
df_test = df_test.drop(columns=cat_features).join(df_test_encoded)

# Đổi tên cột để dễ nhận diện (Thêm hậu tố _encoded)
rename_dict = {col: f"{col}_encoded" for col in cat_features}
df_train = df_train.rename(columns=rename_dict)
df_test = df_test.rename(columns=rename_dict)

print("\n[Thành công] Target Encoding hoàn tất! Dữ liệu mẫu:")
print(df_train[[f"{col}_encoded" for col in cat_features]].head())

--- TARGET ENCODING ---
1. Đang mã hóa biến mục tiêu (Label Encoding)...
2. Đang học quy luật (Fit) và mã hóa tập Train trên 5 cột...
3. Đang áp dụng quy luật (Transform) lên tập Test...

[Thành công] Target Encoding hoàn tất! Dữ liệu mẫu:
   region_encoded  publish_day_of_week_encoded  definition_encoded  \
0        4.058309                     3.997315            3.985874   
1        4.058309                     3.995130            3.985874   
2        4.109259                     3.898098            3.985874   
3        4.109259                     3.898098            3.985874   
4        4.109259                     3.997315            3.985874   

   license_encoded  default_language_encoded  
0         3.920868                  4.194214  
1         3.920868                  3.841369  
2         3.920868                  3.766720  
3         3.920868                  3.766720  
4         3.920868                  3.841369  


In [18]:
print("--- BOOLEAN ENCODING & DATA FORMATTING ---")

# 1. Chuẩn hóa biến logic Boolean (True/False -> 1/0)
print("1. Đang chuẩn hóa các biến Boolean đặc thù sang hệ nhị phân...")
bool_cols_train = df_train.select_dtypes(include=['bool']).columns
df_train[bool_cols_train] = df_train[bool_cols_train].astype(int)

bool_cols_test = df_test.select_dtypes(include=['bool']).columns
df_test[bool_cols_test] = df_test[bool_cols_test].astype(int)


# 2. Định hình không gian đặc trưng số thuần túy (Tách X, y để chuẩn bị train)
print("2. Tiến hành quét sạch các cột văn bản gốc và trích xuất ma trận số...")
cols_to_drop = [
    'crawl_date', 'video_id', 'title', 'channel_title', 'published_at', 
    'category_id', 'category_name', 'thumbnail_url', 'url', 
    'tags', 'tags_source', 'channel_id'
] 

# Hàm an toàn lọc cột thực tế đang tồn tại để tránh crash lỗi KeyError
cols_to_drop_train = [col for col in cols_to_drop if col in df_train.columns]
cols_to_drop_test = [col for col in cols_to_drop if col in df_test.columns]

X_train = df_train.drop(columns=cols_to_drop_train + ['target_category'])
y_train = df_train['target_category'] 

X_test = df_test.drop(columns=cols_to_drop_test + ['target_category'])
y_test = df_test['target_category']

print("\n[Thành công] Ma trận X_train và X_test đã sạch số 100%. Sẵn sàng huấn luyện!")

--- BOOLEAN ENCODING & DATA FORMATTING ---
1. Đang chuẩn hóa các biến Boolean đặc thù sang hệ nhị phân...
2. Tiến hành quét sạch các cột văn bản gốc và trích xuất ma trận số...

[Thành công] Ma trận X_train và X_test đã sạch số 100%. Sẵn sàng huấn luyện!


# 2. Cân bằng trọng số lớp & Huấn luyện mô hình  (Class Weight Balancing & Model Training)

<div style="font-size: 17px;">

Để giải quyết bài toán mất cân bằng dữ liệu nghiêm trọng (danh mục Gaming chiếm áp đảo), nhóm sử dụng chiến lược can thiệp trực tiếp vào thuật toán thay vì sinh thêm dữ liệu giả:
* Class Weight (Trọng số lớp): Kích hoạt tham số *class_weight='balanced'* trong thuật toán. Mô hình sẽ tự động "phạt nặng" khi dự đoán sai các nhóm thiểu số (Education, Science...), ép hệ thống phải tập trung học các nhóm này mà không cần sử dụng kỹ thuật SMOTE.
* Tối giản Pipeline: Khai thác đặc thù của mô hình Cây (rẽ nhánh dựa trên lớn/nhỏ thay vì khoảng cách), lược bỏ hoàn toàn các bước Log Transform và Scaling. Dữ liệu giữ nguyên bản chất gốc, giúp tiết kiệm bộ nhớ RAM và đẩy tốc độ huấn luyện lên mức tối đa.

</div>

In [19]:
print("--- CLASS WEIGHT BALANCING & MODEL TRAINING ---")


# 1. Khởi tạo mô hình Tree-based với Học nhạy cảm chi phí
print("\n[1] Khởi tạo hệ thống LightGBM với class_weight='balanced'...")
lgbm_model = LGBMClassifier(
    class_weight='balanced',  # Bật Cost-Sensitive Learning 
    random_state=42,
    n_estimators=150,   # Cho cây mọc thêm tí để học sâu hơn
    verbose=-1
)

# 2. Huấn luyện (fit) trực tiếp trên dữ liệu gốc + encoded
print("[2] Đang huấn luyện mô hình trên tập Train...")
lgbm_model.fit(X_train, y_train)

# 3. Dự đoán và Kiểm định (Predict & Evaluate)
print("[3] Đang dự đoán trên tập Test và xuất báo cáo...")
y_pred = lgbm_model.predict(X_test)

# Giải mã lại các con số (0, 1, 2...) về tên danh mục gốc
target_names = label_encoder.classes_

print("\n=== BÁO CÁO KIỂM ĐỊNH MÔ HÌNH (CLASSIFICATION REPORT) ===")
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

--- CLASS WEIGHT BALANCING & MODEL TRAINING ---

[1] Khởi tạo hệ thống LightGBM với class_weight='balanced'...
[2] Đang huấn luyện mô hình trên tập Train...
[3] Đang dự đoán trên tập Test và xuất báo cáo...

=== BÁO CÁO KIỂM ĐỊNH MÔ HÌNH (CLASSIFICATION REPORT) ===
                      precision    recall  f1-score   support

              Comedy       1.00      0.80      0.89         5
           Education       1.00      1.00      1.00         2
       Entertainment       0.99      0.93      0.96       196
    Film & Animation       0.83      0.89      0.86        28
              Gaming       0.97      0.99      0.98       773
               Music       0.91      0.95      0.93       156
              Others       1.00      1.00      1.00         3
      People & Blogs       0.82      0.69      0.75        48
Science & Technology       1.00      0.80      0.89         5
              Sports       1.00      0.50      0.67         4
     Travel & Events       1.00      1.00      1.00

In [20]:
print("--- XEM KẾT QUẢ DỰ ĐOÁN THỰC TẾ TRÊN TỪNG VIDEO ---")

# 1. Tạo một bảng (DataFrame) để đối chiếu
results_df = pd.DataFrame({
    'video_id': df_test['video_id'],         # Lấy lại ID video gốc
    'title': df_test['title'],               # Lấy lại tiêu đề video gốc
    'Actual_Category': df_test['category_name'], # Danh mục THẬT của YouTube
    
    # Giải mã y_pred từ số (0, 1, 2...) ngược trở lại thành chữ (Gaming, Music...)
    'Predicted_Category': label_encoder.inverse_transform(y_pred) 
})

# 2. Tạo thêm một cột đánh giá xem máy đoán Đúng hay Sai
results_df['Is_Correct'] = results_df['Actual_Category'] == results_df['Predicted_Category']

# 3. In ra 10 video ngẫu nhiên để biểu diễn
print("\n[Mẫu 10 video ngẫu nhiên] Thực tế vs Dự đoán:")
print(results_df[['title', 'Actual_Category', 'Predicted_Category', 'Is_Correct']].sample(10))

# 4. (Tùy chọn) In ra những video máy ĐOÁN SAI để phân tích nguyên nhân
print("\n[Phân tích ca khó] 10 video mà máy đoán SAI:")
wrong_predictions = results_df[results_df['Is_Correct'] == False]
print(wrong_predictions[['title', 'Actual_Category', 'Predicted_Category']].head(10))

--- XEM KẾT QUẢ DỰ ĐOÁN THỰC TẾ TRÊN TỪNG VIDEO ---

[Mẫu 10 video ngẫu nhiên] Thực tế vs Dự đoán:
                                                 title Actual_Category  \
859  The 23,300 IQ VIGILANTE vs EVIL GUESSER showdo...          Gaming   
479  THIS ISNT FAIR FOR NRG?! (FNS Reacts To NRG vs...          Gaming   
863           NEW MYSTERY MODE in KNOCKOUT is CHAOS...  People & Blogs   
937                         WWE Friday Night SmackDown   Entertainment   
331  PRX vs KRX / RRQ vs DFM - VCT Pacific - Stage ...          Gaming   
529                    27.03.27 - ACCOR ARENA - PARIS            Music   
6    TÌM EM CÂU VÍ SÔNG LAM | LK Dân Ca Xứ Nghệ Hay...           Music   
424     Hungry Shark World speedruns are pure insanity          Gaming   
463  Chris Brown - Fallin' (Official Video) ft. Leo...           Music   
951  Maximum Pleasure Guaranteed — Official Teaser ...   Entertainment   

    Predicted_Category  Is_Correct  
859             Gaming        True  
479         